In [ ]:
from astropy.io import fits
from astropy import wcs as WCS
import numpy as np
import matplotlib.pyplot as plt
#import math
#from mpl_toolkits.axes_grid1 import make_axes_locatable
#from matplotlib.colors import LogNorm
import os
import subprocess
#import importlib as imp

## Read in Q,U,I file, make 3D headers

In [ ]:
hdu = fits.open('/srv/data/gmims/gmims-hbn/GMIMS-HBN_v1_gal_car_freq_IQU.fits')
hdr4D = hdu[0].header

hdr3D = hdr4D.copy()
del hdr3D['NAXIS4']
del hdr3D['CRPIX4']
del hdr3D['CDELT4']
del hdr3D['CTYPE4']
del hdr3D['CRVAL4']
hdr3D['NAXIS'] = 3
hdr3D['WCSAXES'] = 3
print(repr(hdr3D))

I = hdu[0].data[0]
Q = hdu[0].data[1]
U = hdu[0].data[2]

print(Q.shape)

freq = np.linspace(hdr4D['CRVAL3'],
                   hdr4D['CRVAL3']+hdr4D['CDELT3']*(hdr4D['NAXIS3']-1),
                   hdr4D['NAXIS3'])


## Make indices of bad frequencies

In [ ]:
idx_bad1 = np.array(range(0,20))
idx_bad2 = np.array(range(79,83))
idx_bad3 = np.array(range(97,100))
idx_bad4 = np.array(range(206,266))
idx_bad5 = np.array(range(270,281))
idx_bad6 = np.array(range(286,300))
idx_bad7 = np.array(range(336,340))
idx_bad8 = np.array(range(343,354))
idx_bad9 = np.array(range(376,409))

idx_bad = np.array(np.concatenate([idx_bad1,idx_bad2,idx_bad3,
                                  idx_bad4,idx_bad5,idx_bad6,
                                  idx_bad7,idx_bad8,idx_bad9]))
print(freq[idx_bad]/1e6)

## NaN the bad data channels

In [ ]:
Q[idx_bad,:,:] = np.nan
U[idx_bad,:,:] = np.nan
I[idx_bad,:,:] = np.nan

## Write out separate 3D Q, U and I files

In [ ]:
fits.writeto('/srv/aordog/cgps_gmims_data/gmims_hbn_Q.fits',Q,hdr3D,overwrite=True)
fits.writeto('/srv/aordog/cgps_gmims_data/gmims_hbn_U.fits',U,hdr3D,overwrite=True)
fits.writeto('/srv/aordog/cgps_gmims_data/gmims_hbn_I.fits',I,hdr3D,overwrite=True)

## Do the RM synthesis

In [ ]:
!rmsynth3d /srv/aordog/cgps_gmims_data/gmims_hbn_Q.fits \
           /srv/aordog/cgps_gmims_data/gmims_hbn_U.fits \
           /srv/aordog/cgps_gmims_data/gmims_freq.txt \
           -t -l 500 -d 2 -v

In [ ]:
!rmtools_peakfitcube -h